# Results for model: glm-4.6

In [ ]:
import polars as pl
import xgboost as xgb
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import joblib
import warnings
warnings.filterwarnings('ignore')

def load_data():
    """Load training data using Polars"""
    return pl.read_parquet('./jane-street-real-time-market-data-forecasting/train.parquet')

def calculate_top_quantile(df, window_size=5):
    """Calculate TOP_QUANTILE feature for feature_00 relative to responder_6"""
    df = df.sort('date_id')
    
    # Get unique date_ids
    unique_dates = df['date_id'].unique().to_list()
    
    # Initialize result column
    top_quantile_flags = []
    
    for i, date in enumerate(unique_dates):
        # Get rolling window of dates
        start_idx = max(0, i - window_size // 2)
        end_idx = min(len(unique_dates), i + window_size // 2 + 1)
        window_dates = unique_dates[start_idx:end_idx]
        
        # Filter data for current window
        window_data = df.filter(pl.col('date_id').is_in(window_dates))
        
        # Calculate correlation between feature_00 and responder_6
        corr = window_data.select(
            pl.corr('feature_00', 'responder_6')
        ).item()
        
        # If positive correlation, top quantile means high feature_00 values
        # If negative correlation, top quantile means low feature_00 values
        if corr > 0:
            threshold = window_data['feature_00'].quantile(0.85)
            current_data = df.filter(pl.col('date_id') == date)
            flags = (current_data['feature_00'] >= threshold).to_list()
        else:
            threshold = window_data['feature_00'].quantile(0.15)
            current_data = df.filter(pl.col('date_id') == date)
            flags = (current_data['feature_00'] <= threshold).to_list()
        
        top_quantile_flags.extend(flags)
    
    return df.with_columns(
        pl.Series('TOP_QUANTILE', top_quantile_flags).cast(pl.Int8)
    )

def feature_engineering(df):
    """Perform feature engineering"""
    # Calculate TOP_QUANTILE feature
    df = calculate_top_quantile(df)
    
    # Fill missing values
    numeric_cols = [col for col in df.columns if col.startswith('feature_')]
    df = df.with_columns([
        pl.col(col).fill_null(pl.col(col).mean()) for col in numeric_cols
    ])
    
    # Add interaction features
    df = df.with_columns([
        (pl.col('feature_00') * pl.col('feature_01')).alias('feat_00_01'),
        (pl.col('feature_02') * pl.col('feature_03')).alias('feat_02_03'),
        (pl.col('feature_04') * pl.col('feature_05')).alias('feat_04_05'),
    ])
    
    return df

def train_model(df):
    """Train XGBoost model"""
    # Select features
    feature_cols = [col for col in df.columns if col.startswith('feature_') or col == 'TOP_QUANTILE']
    feature_cols.extend(['feat_00_01', 'feat_02_03', 'feat_04_05'])
    
    # Prepare data
    X = df.select(feature_cols).to_pandas()
    y = df['responder_6'].to_pandas()
    
    # Split data
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    # Train XGBoost model
    params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'max_depth': 6,
        'learning_rate': 0.1,
        'n_estimators': 1000,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'random_state': 42,
        'n_jobs': -1
    }
    
    model = xgb.XGBRegressor(**params)
    
    # Train with early stopping
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        early_stopping_rounds=50,
        verbose=False
    )
    
    # Evaluate
    val_pred = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, val_pred))
    print(f'Validation RMSE: {rmse:.6f}')
    
    return model, feature_cols

def main():
    # Load data
    print("Loading data...")
    df = load_data()
    print(f"Data shape: {df.shape}")
    
    # Feature engineering
    print("Performing feature engineering...")
    df = feature_engineering(df)
    
    # Train model
    print("Training model...")
    model, feature_cols = train_model(df)
    
    # Save model and feature list
    print("Saving model...")
    joblib.dump(model, 'xgb_model.pkl')
    joblib.dump(feature_cols, 'feature_cols.pkl')
    
    print("Training complete!")

if __name__ == "__main__":
    main()